# Temporal Cross-Validation

Standard k-fold cross-validation **does not work** for time series data because it ignores
the temporal ordering of observations. Randomly shuffling time series data breaks the
autocorrelation structure and causes **data leakage** — the model sees future information
during training.

This notebook covers temporal cross-validation strategies that respect the time ordering:

- **Expanding window** (growing training set)
- **Rolling window** (fixed-size training set)
- **Multi-step ahead** cross-validation

We use Brazilian macroeconomic data and `forecastbox` to demonstrate each approach.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from forecastbox.metrics import mae, rmse
from forecastbox.cv import expanding_window_cv, rolling_window_cv
from forecastbox.auto._baselines import NaiveBaseline

import sys
sys.path.insert(0, "..")
from utils.helpers import load_macro_brazil, load_macro_us

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 4)

## 1. The Problem with Standard CV

In standard k-fold CV, data is randomly split into train/test folds:

```
Standard k-fold CV (WRONG for time series):
──────────────────────────────────────────────
Fold 1: [TEST] [train] [train] [TEST] [train]
Fold 2: [train] [TEST] [train] [train] [TEST]
Fold 3: [train] [train] [TEST] [train] [TEST]
                 ↑                ↑
           Future data used     Past data
           to train model!      in test set!
                 = DATA LEAKAGE
```

In temporal CV, we always train on the past and test on the future:

```
Temporal CV (CORRECT):
──────────────────────────────────────────────
Fold 1: [train train train] [test]
Fold 2: [train train train train] [test]
Fold 3: [train train train train train] [test]
             Past → → → → → → Future
```

The training set always precedes the test set in time.

In [ ]:
# Visual comparison: standard CV vs temporal CV
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

n_points = 20
n_folds = 4

# Standard CV (random splits) - WRONG
ax = axes[0]
ax.set_title("Standard k-fold CV (WRONG for time series)", fontsize=13, fontweight="bold")
np.random.seed(42)
for fold in range(n_folds):
    colors = ["#2196F3"] * n_points  # blue = train
    test_idx = np.random.choice(n_points, size=n_points // n_folds, replace=False)
    for idx in test_idx:
        colors[idx] = "#F44336"  # red = test
    for i in range(n_points):
        ax.barh(fold, 1, left=i, color=colors[i], edgecolor="white", linewidth=0.5)
ax.set_yticks(range(n_folds))
ax.set_yticklabels([f"Fold {i+1}" for i in range(n_folds)])
ax.set_xlabel("Time →")

# Temporal CV (expanding window) - CORRECT
ax = axes[1]
ax.set_title("Temporal CV - Expanding Window (CORRECT)", fontsize=13, fontweight="bold")
initial = 8
horizon = 3
for fold in range(n_folds):
    train_end = initial + fold * 2
    for i in range(n_points):
        if i < train_end:
            color = "#2196F3"  # blue = train
        elif i < train_end + horizon:
            color = "#F44336"  # red = test
        else:
            color = "#E0E0E0"  # gray = unused
        ax.barh(fold, 1, left=i, color=color, edgecolor="white", linewidth=0.5)
ax.set_yticks(range(n_folds))
ax.set_yticklabels([f"Fold {i+1}" for i in range(n_folds)])
ax.set_xlabel("Time →")

# Legend
legend_patches = [
    mpatches.Patch(color="#2196F3", label="Train"),
    mpatches.Patch(color="#F44336", label="Test"),
    mpatches.Patch(color="#E0E0E0", label="Unused"),
]
fig.legend(handles=legend_patches, loc="lower center", ncol=3, fontsize=11)
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

## 2. Expanding Window

In expanding window CV, the training set **grows** with each fold while the forecast
horizon remains fixed:

```
Expanding Window CV:
──────────────────────────────────────────
Fold 1: [=====train=====] [test]
Fold 2: [======train======] [test]
Fold 3: [=======train=======] [test]
Fold 4: [========train========] [test]
```

**Advantages:**
- Uses all available past data for training
- More training data in later folds → potentially better models

**Disadvantages:**
- If the data-generating process changes (structural breaks), old data may hurt

In [ ]:
# Load data
df_brazil = load_macro_brazil()
gdp = df_brazil["gdp_growth"].dropna()
print(f"GDP growth series: {len(gdp)} observations")
print(f"Period: {gdp.index[0].strftime('%Y-%m')} to {gdp.index[-1].strftime('%Y-%m')}")

# Define a naive model function for CV
def naive_model_fn(train: pd.Series) -> np.ndarray:
    """Naive forecast: repeat last value for all horizons."""
    return np.full(12, train.iloc[-1])  # h=12 max

# Expanding window CV
results_expanding = expanding_window_cv(
    data=gdp,
    model_fn=naive_model_fn,
    initial_window=60,   # start with 5 years of data
    horizon=12,          # forecast 12 months ahead
    step=6,              # move forward 6 months between folds
    verbose=True,
)

print("\n" + results_expanding.summary())

In [ ]:
# Visualize error by horizon
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

results_expanding.plot_errors(ax=axes[0])
axes[0].set_title("Expanding Window: Error by Horizon")

results_expanding.plot_forecast_vs_actual(ax=axes[1])
axes[1].set_title("Expanding Window: Last Fold")

plt.tight_layout()
plt.show()

## 3. Rolling Window (Fixed Size)

In rolling window CV, the training set has a **fixed size** and slides through the data:

```
Rolling Window CV (fixed window):
──────────────────────────────────────────
Fold 1:     [===train===] [test]
Fold 2:       [===train===] [test]
Fold 3:         [===train===] [test]
Fold 4:           [===train===] [test]
```

**Advantages:**
- Adapts better to structural changes (drops old, potentially irrelevant data)
- Consistent training set size → consistent model complexity

**Disadvantages:**
- Discards older data that may still be informative
- Needs to choose window size (too small = noisy, too large = slow to adapt)

**Trade-off:** Use expanding window when you believe the full history is relevant;
use rolling window when you suspect structural breaks or regime changes.

In [ ]:
# Rolling window CV with fixed window of 60 months
results_rolling = rolling_window_cv(
    data=gdp,
    model_fn=naive_model_fn,
    window=60,           # fixed training window: 5 years
    horizon=12,          # forecast 12 months ahead
    step=6,              # move forward 6 months between folds
    verbose=True,
)

print(f"\nNumber of folds: {results_rolling.n_folds}")
print("\n" + results_rolling.summary())

## 4. Comparing Strategies

Let's compare expanding window vs rolling window side by side for the naive forecast
on GDP growth. We look at both MAE and RMSE across folds.

In [ ]:
# Compare expanding vs rolling
expanding_mae = results_expanding.metrics_overall.get("mae", np.nan)
expanding_rmse = results_expanding.metrics_overall.get("rmse", np.nan)

rolling_means = results_rolling.mean_metrics()
rolling_mae = rolling_means.get("mae", np.nan)
rolling_rmse = rolling_means.get("rmse", np.nan)

comparison = pd.DataFrame({
    "Expanding Window": {"MAE": expanding_mae, "RMSE": expanding_rmse,
                         "Folds": results_expanding.n_folds},
    "Rolling Window (w=60)": {"MAE": rolling_mae, "RMSE": rolling_rmse,
                              "Folds": results_rolling.n_folds},
}).T
print("Comparison: Expanding vs Rolling Window")
print("=" * 50)
print(comparison.round(4))

# Visual comparison by fold
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Expanding: error by horizon
h = results_expanding.metrics_by_horizon["horizon"]
axes[0].plot(h, results_expanding.metrics_by_horizon["mae"], "o-", label="Expanding")

# Rolling: fold-level MAE
rolling_fold_mae = [f.metrics["mae"] for f in results_rolling.folds]
axes[1].bar(range(1, len(rolling_fold_mae) + 1), rolling_fold_mae, alpha=0.7)
axes[1].set_xlabel("Fold")
axes[1].set_ylabel("MAE")
axes[1].set_title("Rolling Window: MAE by Fold")
axes[1].grid(True, alpha=0.3)

axes[0].set_title("Expanding Window: MAE by Horizon")
axes[0].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Multi-step Ahead CV

Often we want to evaluate forecast accuracy at **multiple horizons** (e.g., 1, 3, 6, and 12
months ahead). This helps answer: "How quickly does forecast accuracy degrade as the
horizon increases?"

We run expanding window CV with different horizons and compare the results.

In [ ]:
# Multi-step ahead: evaluate at horizons h=1, 3, 6, 12
horizons = [1, 3, 6, 12]
multi_results = {}

for h in horizons:
    def model_fn(train, h=h):
        return np.full(h, train.iloc[-1])

    cv_result = expanding_window_cv(
        data=gdp,
        model_fn=model_fn,
        initial_window=60,
        horizon=h,
        step=6,
    )
    multi_results[h] = cv_result

# Summary table
rows = []
for h, res in multi_results.items():
    rows.append({
        "Horizon (months)": h,
        "MAE": res.metrics_overall.get("mae", np.nan),
        "RMSE": res.metrics_overall.get("rmse", np.nan),
        "Folds": res.n_folds,
    })

multi_df = pd.DataFrame(rows)
print("Multi-step Ahead CV: Naive Forecast for GDP Growth")
print("=" * 55)
print(multi_df.to_string(index=False))

# Plot MAE by horizon
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(multi_df["Horizon (months)"], multi_df["MAE"], "o-", linewidth=2, markersize=8)
ax.set_xlabel("Forecast Horizon (months)", fontsize=12)
ax.set_ylabel("MAE", fontsize=12)
ax.set_title("Forecast Error Increases with Horizon", fontsize=14)
ax.set_xticks(horizons)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Exercise 1: Implement rolling CV for inflation

Use `rolling_window_cv` from forecastbox to evaluate a naive forecast for Brazil's
inflation series. Use a window of 60 months, horizon of 12, and step of 3.

In [ ]:
# TODO: Exercise 1 - Rolling CV for Brazil inflation with window=60

## Exercise 2: Compare expanding vs rolling for exchange_rate

Load the Brazil dataset's `exchange_rate` series and compare expanding window CV
vs rolling window CV for a naive forecast. Which strategy gives better results? Why?

In [ ]:
# TODO: Exercise 2 - Compare both strategies